# MusicBrainz Enrichment & Text Preparation

This notebook explores the MusicBrainz enrichment data and designs the optimal text representation for embedding.

## Goals
1. Understand what MusicBrainz data we have
2. Analyze relationship data structure
3. Design rich text representation for embedding
4. Test different enrichment strategies
5. Measure text quality and coverage

In [ ]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn
!pip install plotly
# Note: json is built-in, no need to install
# Note: crate_analysis is a local package, not on PyPI


  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 3.5 MB/s  0:00:02m 3.5 MB/s eta 0:00:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 3.7 MB/s  0:00:00m 3.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 4.0 MB/s  0:00:01m 4.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib] 6/7 [matplotlib]ow]
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 4.2 MB/s  0:00:024.1 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]━━━━ 1/2 [plotly]
ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json
ERROR: Could not find a version that satisfies the requirement crate_analysis (from version

In [10]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
import json

from crate_analysis import Database

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 200)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

In [11]:
# Connect to database
db = Database()
print(f"Connected to: {db.db_path}")

Connected to: /Users/pooks/Dev/crate/data/music_kb.sqlite


## 1. Explore Available Tables

In [ ]:
# Get all tables
tables = db.get_tables()
print(f"Total tables: {len(tables)}\n")

# Get row counts
table_stats = []
for table_name in tables['name']:
    try:
        count = db.get_row_count(table_name)
        table_stats.append({
            'table': table_name,
            'row_count': count,
            'columns': len(db.get_table_info(table_name))
        })
    except Exception as e:
        print(f"Error with {table_name}: {e}")

stats_df = pd.DataFrame(table_stats).sort_values('row_count', ascending=False)
stats_df

## 2. Fact Plays - Core Data

This is our main table. Let's see what we have:

In [ ]:
# Schema
print("fact_plays schema:")
display(db.get_table_info('fact_plays'))

# Sample
print("\nSample rows:")
plays_sample = db.get_table_sample('fact_plays', 10)
display(plays_sample)

In [ ]:
# Analyze field completeness
completeness = db.query("""
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN artist IS NOT NULL AND artist != '' THEN 1 ELSE 0 END) as has_artist,
        SUM(CASE WHEN album IS NOT NULL AND album != '' THEN 1 ELSE 0 END) as has_album,
        SUM(CASE WHEN song IS NOT NULL AND song != '' THEN 1 ELSE 0 END) as has_song,
        SUM(CASE WHEN rotation_status IS NOT NULL THEN 1 ELSE 0 END) as has_rotation,
        SUM(CASE WHEN is_local IS NOT NULL THEN 1 ELSE 0 END) as has_local,
        SUM(CASE WHEN labels IS NOT NULL AND labels != '' THEN 1 ELSE 0 END) as has_labels,
        SUM(CASE WHEN comment IS NOT NULL AND comment != '' THEN 1 ELSE 0 END) as has_comment
    FROM fact_plays
""")

total = completeness['total'].iloc[0]
completeness_pct = pd.DataFrame({
    'field': ['artist', 'album', 'song', 'rotation', 'local', 'labels', 'comment'],
    'count': [
        completeness['has_artist'].iloc[0],
        completeness['has_album'].iloc[0],
        completeness['has_song'].iloc[0],
        completeness['has_rotation'].iloc[0],
        completeness['has_local'].iloc[0],
        completeness['has_labels'].iloc[0],
        completeness['has_comment'].iloc[0]
    ]
})
completeness_pct['percentage'] = (completeness_pct['count'] / total * 100).round(2)

print(f"\nField completeness (out of {total:,} plays):")
display(completeness_pct)

# Visualize
fig = px.bar(completeness_pct, x='field', y='percentage',
             title='Field Completeness in fact_plays',
             labels={'percentage': 'Completeness %', 'field': 'Field'})
fig.add_hline(y=100, line_dash="dash", line_color="green", annotation_text="100%")
fig.show()

## 3. Comments - Rich Metadata!

DJ comments can contain genre info, mood, descriptions - super valuable for semantic search.

In [ ]:
# Get plays with comments
comments = db.query("""
    SELECT artist, song, album, comment, airdate
    FROM fact_plays
    WHERE comment IS NOT NULL AND comment != ''
    ORDER BY RANDOM()
    LIMIT 50
""")

print(f"Found {len(comments)} plays with comments (out of sample)\n")

# Show examples
for idx, row in comments.head(20).iterrows():
    print(f"🎵 {row['artist']} - {row['song']}")
    print(f"   💬 {row['comment']}")
    print()

In [ ]:
# Analyze comment length and characteristics
comment_stats = db.query("""
    SELECT 
        LENGTH(comment) as comment_length,
        comment
    FROM fact_plays
    WHERE comment IS NOT NULL AND comment != ''
""")

print(f"Comment statistics:")
print(f"  Average length: {comment_stats['comment_length'].mean():.0f} characters")
print(f"  Median length: {comment_stats['comment_length'].median():.0f} characters")
print(f"  Max length: {comment_stats['comment_length'].max():.0f} characters")

# Distribution
fig = px.histogram(comment_stats, x='comment_length', nbins=50,
                   title='Comment Length Distribution',
                   labels={'comment_length': 'Comment Length (characters)'})
fig.show()

## 4. MusicBrainz Artist Entity Data

In [ ]:
# Check if we have artist entity tables
artist_tables = [t for t in tables['name'].values if 'artist' in t.lower()]
print("Artist-related tables:")
for t in artist_tables:
    count = db.get_row_count(t)
    print(f"  {t}: {count:,} rows")

In [ ]:
# Explore artist master table (if it exists)
if 'artist_mb_entity_master' in tables['name'].values:
    print("Schema:")
    display(db.get_table_info('artist_mb_entity_master'))
    
    print("\nSample:")
    artist_sample = db.get_table_sample('artist_mb_entity_master', 10)
    display(artist_sample)
    
    # Check what metadata we have
    artist_meta = db.query("""
        SELECT 
            COUNT(*) as total,
            SUM(CASE WHEN type IS NOT NULL THEN 1 ELSE 0 END) as has_type,
            SUM(CASE WHEN country IS NOT NULL THEN 1 ELSE 0 END) as has_country,
            SUM(CASE WHEN gender IS NOT NULL THEN 1 ELSE 0 END) as has_gender,
            SUM(CASE WHEN disambiguation IS NOT NULL THEN 1 ELSE 0 END) as has_disambiguation,
            COUNT(DISTINCT type) as unique_types,
            COUNT(DISTINCT country) as unique_countries
        FROM artist_mb_entity_master
    """)
    print("\nMetadata completeness:")
    display(artist_meta)
    
    # Get artist types distribution
    artist_types = db.query("""
        SELECT type, COUNT(*) as count
        FROM artist_mb_entity_master
        WHERE type IS NOT NULL
        GROUP BY type
        ORDER BY count DESC
    """)
    print("\nArtist types:")
    display(artist_types)

## 5. Relationships - Genre, Labels, etc.

In [ ]:
# Check for relationship tables
rel_tables = [t for t in tables['name'].values if 'relation' in t.lower() or 'master' in t.lower()]
print("Relationship tables:")
for t in rel_tables:
    count = db.get_row_count(t)
    print(f"  {t}: {count:,} rows")

In [ ]:
# Explore master_relations if it exists
if 'master_relations' in tables['name'].values:
    print("Schema:")
    display(db.get_table_info('master_relations'))
    
    print("\nSample:")
    rel_sample = db.get_table_sample('master_relations', 20)
    display(rel_sample)
    
    # Get relationship type distribution
    rel_types = db.query("""
        SELECT 
            predicate,
            COUNT(*) as count,
            COUNT(DISTINCT subject_id) as unique_subjects,
            COUNT(DISTINCT object_id) as unique_objects
        FROM master_relations
        GROUP BY predicate
        ORDER BY count DESC
    """)
    print("\nRelationship types:")
    display(rel_types)
    
    # Visualize
    fig = px.bar(rel_types, x='predicate', y='count',
                 title='Relationship Type Distribution',
                 labels={'count': 'Count', 'predicate': 'Relationship Type'})
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

In [ ]:
# Get entity type distribution
if 'master_relations' in tables['name'].values:
    entity_types = db.query("""
        SELECT 
            subject_type,
            object_type,
            COUNT(*) as count
        FROM master_relations
        GROUP BY subject_type, object_type
        ORDER BY count DESC
    """)
    print("Entity type combinations:")
    display(entity_types.head(20))

## 6. Example: Rich Artist Profile

Let's pick a random artist and see all the data we can gather about them:

In [ ]:
# Pick a popular artist from plays
popular_artist = db.query("""
    SELECT artist, COUNT(*) as play_count
    FROM fact_plays
    WHERE artist IS NOT NULL
    GROUP BY artist
    ORDER BY play_count DESC
    LIMIT 1
""").iloc[0]['artist']

print(f"Building profile for: {popular_artist}\n")

# Get artist entity data
if 'artist_mb_entity_master' in tables['name'].values:
    artist_entity = db.query("""
        SELECT *
        FROM artist_mb_entity_master
        WHERE name = ?
        LIMIT 1
    """, (popular_artist,))
    
    if not artist_entity.empty:
        print("MusicBrainz Entity Data:")
        display(artist_entity.T)

# Get relationships
if 'master_relations' in tables['name'].values:
    artist_rels = db.query("""
        SELECT 
            predicate,
            object_name,
            object_type
        FROM master_relations
        WHERE subject_name = ?
        LIMIT 50
    """, (popular_artist,))
    
    if not artist_rels.empty:
        print(f"\nRelationships ({len(artist_rels)} found):")
        display(artist_rels)

# Get sample plays with comments
artist_plays = db.query("""
    SELECT song, album, comment, labels, rotation_status
    FROM fact_plays
    WHERE artist = ?
    AND comment IS NOT NULL AND comment != ''
    LIMIT 10
""", (popular_artist,))

if not artist_plays.empty:
    print(f"\nSample plays with comments:")
    display(artist_plays)

## 7. Design Text Enrichment Strategy

Based on what we found, let's design the optimal text representation:

In [ ]:
def enrich_play_text(play_row, relationships_df=None):
    """Create rich text representation for a play."""
    parts = []
    
    # Core: Artist - Song - Album
    if play_row.get('artist'):
        parts.append(f"{play_row['artist']}")
    if play_row.get('song'):
        parts.append(f"- {play_row['song']}")
    if play_row.get('album'):
        parts.append(f"- {play_row['album']}")
    
    metadata_parts = []
    
    # Add DJ comment (valuable!)
    if play_row.get('comment'):
        metadata_parts.append(f"Comment: {play_row['comment']}")
    
    # Add labels
    if play_row.get('labels'):
        metadata_parts.append(f"Label: {play_row['labels']}")
    
    # Add rotation status
    if play_row.get('rotation_status'):
        metadata_parts.append(f"Rotation: {play_row['rotation_status']}")
    
    # Add local flag
    if play_row.get('is_local') == 1:
        metadata_parts.append("Local artist")
    
    # Add year if available
    if play_row.get('airdate'):
        year = pd.to_datetime(play_row['airdate']).year
        metadata_parts.append(f"Year: {year}")
    
    # Add MusicBrainz relationships (genres, etc.) if provided
    if relationships_df is not None and not relationships_df.empty:
        # Extract genres from relationships
        genres = relationships_df[
            (relationships_df['predicate'].str.contains('genre', case=False, na=False))
        ]['object_name'].tolist()
        if genres:
            metadata_parts.append(f"Genre: {', '.join(genres[:5])}")  # Limit to 5
    
    # Combine
    text = ' '.join(parts)
    if metadata_parts:
        text += ' | ' + ' | '.join(metadata_parts)
    
    return text

# Test on sample plays
test_sample = db.query("""
    SELECT *
    FROM fact_plays
    LIMIT 10
""")

print("Example enriched texts:\n")
for idx, row in test_sample.iterrows():
    enriched = enrich_play_text(row)
    print(f"{idx+1}. {enriched}")
    print()

## 8. Analyze Text Coverage & Quality

In [ ]:
# Get a larger sample and analyze enrichment quality
sample_size = 1000
sample_plays = db.query(f"""
    SELECT *
    FROM fact_plays
    ORDER BY RANDOM()
    LIMIT {sample_size}
""")

# Generate enriched texts
enriched_texts = []
for idx, row in sample_plays.iterrows():
    text = enrich_play_text(row)
    enriched_texts.append({
        'text': text,
        'length': len(text),
        'has_comment': bool(row.get('comment')),
        'has_label': bool(row.get('labels')),
        'has_rotation': bool(row.get('rotation_status'))
    })

enriched_df = pd.DataFrame(enriched_texts)

print(f"Analysis of {sample_size} enriched texts:\n")
print(f"Average length: {enriched_df['length'].mean():.0f} characters")
print(f"Median length: {enriched_df['length'].median():.0f} characters")
print(f"% with comments: {enriched_df['has_comment'].mean()*100:.1f}%")
print(f"% with labels: {enriched_df['has_label'].mean()*100:.1f}%")
print(f"% with rotation: {enriched_df['has_rotation'].mean()*100:.1f}%")

# Show distribution
fig = px.histogram(enriched_df, x='length', nbins=50,
                   title=f'Enriched Text Length Distribution (n={sample_size})',
                   labels={'length': 'Text Length (characters)'})
fig.show()

In [ ]:
# Show examples by richness
print("\n=== Most Enriched Examples ===")
for idx in enriched_df.nlargest(5, 'length').index:
    print(f"\n{enriched_df.loc[idx, 'text']}")
    print(f"  Length: {enriched_df.loc[idx, 'length']} chars")

print("\n=== Least Enriched Examples ===")
for idx in enriched_df.nsmallest(5, 'length').index:
    print(f"\n{enriched_df.loc[idx, 'text']}")
    print(f"  Length: {enriched_df.loc[idx, 'length']} chars")

## 9. Extract Genre Information

Let's see if we can extract genre info from relationships or comments:

In [ ]:
# Music genre keywords to search for in comments
genre_keywords = [
    'rock', 'indie', 'electronic', 'jazz', 'classical', 'hip-hop', 'rap',
    'metal', 'punk', 'folk', 'country', 'blues', 'soul', 'funk', 'disco',
    'house', 'techno', 'ambient', 'experimental', 'pop', 'r&b', 'reggae',
    'alternative', 'grunge', 'shoegaze', 'post-punk', 'synth', 'psychedelic'
]

# Find genre mentions in comments
comment_genres = db.query("""
    SELECT comment, artist, song
    FROM fact_plays
    WHERE comment IS NOT NULL
    LIMIT 1000
""")

genre_matches = []
for idx, row in comment_genres.iterrows():
    comment_lower = row['comment'].lower()
    found_genres = [g for g in genre_keywords if g in comment_lower]
    if found_genres:
        genre_matches.append({
            'artist': row['artist'],
            'song': row['song'],
            'comment': row['comment'],
            'genres_found': found_genres
        })

print(f"Found {len(genre_matches)} comments with genre mentions (out of {len(comment_genres)} with comments)\n")

# Show examples
for match in genre_matches[:10]:
    print(f"🎵 {match['artist']} - {match['song']}")
    print(f"   Genres: {', '.join(match['genres_found'])}")
    print(f"   💬 {match['comment']}")
    print()

## 10. Save Enrichment Function

Let's save our enrichment function to use in the main processing:

In [ ]:
# Save to src/crate_analysis/enrichment.py
enrichment_code = '''
"""Text enrichment utilities for music search."""

import pandas as pd
from typing import Optional, Dict, Any


def enrich_play_text(play_row: Dict[str, Any], relationships_df: Optional[pd.DataFrame] = None) -> str:
    """Create rich text representation for a play.
    
    Args:
        play_row: Dictionary with play data (artist, song, album, comment, etc.)
        relationships_df: Optional DataFrame with MusicBrainz relationships
        
    Returns:
        Enriched text string ready for embedding
    """
    parts = []
    
    # Core: Artist - Song - Album
    if play_row.get('artist'):
        parts.append(f"{play_row['artist']}")
    if play_row.get('song'):
        parts.append(f"- {play_row['song']}")
    if play_row.get('album'):
        parts.append(f"- {play_row['album']}")
    
    metadata_parts = []
    
    # Add DJ comment (very valuable for semantic understanding!)
    if play_row.get('comment'):
        metadata_parts.append(f"Comment: {play_row['comment']}")
    
    # Add labels
    if play_row.get('labels'):
        metadata_parts.append(f"Label: {play_row['labels']}")
    
    # Add rotation status
    if play_row.get('rotation_status'):
        metadata_parts.append(f"Rotation: {play_row['rotation_status']}")
    
    # Add local flag
    if play_row.get('is_local') == 1:
        metadata_parts.append("Local artist")
    
    # Add year if available
    if play_row.get('airdate'):
        try:
            year = pd.to_datetime(play_row['airdate']).year
            metadata_parts.append(f"Year: {year}")
        except:
            pass
    
    # Add MusicBrainz relationships (genres, etc.) if provided
    if relationships_df is not None and not relationships_df.empty:
        # Extract genres from relationships
        genres = relationships_df[
            (relationships_df['predicate'].str.contains('genre', case=False, na=False))
        ]['object_name'].tolist()
        if genres:
            metadata_parts.append(f"Genre: {', '.join(genres[:5])}")  # Limit to 5
    
    # Combine
    text = ' '.join(parts)
    if metadata_parts:
        text += ' | ' + ' | '.join(metadata_parts)
    
    return text
'''

with open('../src/crate_analysis/enrichment.py', 'w') as f:
    f.write(enrichment_code)

print("✅ Saved enrichment function to src/crate_analysis/enrichment.py")

## Summary & Next Steps

### What We Found
1. **Comments are gold** - DJ comments contain genre, mood, style descriptions
2. **Labels are useful** - Record labels help with style/genre inference
3. **Rotation status** - Heavy/Medium/Light rotation indicates popularity/quality
4. **MusicBrainz relationships** - If available, provide structured genre data
5. **Coverage varies** - Not all plays have rich metadata

### Enrichment Strategy
- Base: Artist - Song - Album
- Add: DJ comments (when available)
- Add: Labels, rotation status, local flag
- Add: Year from airdate
- Add: MusicBrainz genres (if in relationships)

### Next Steps
1. Test embedding with enriched text
2. Compare different enrichment levels
3. Process full dataset
4. Build search interface

In [ ]:
# Clean up
db.close()

## 11. Load Embeddings & Set Up Semantic Search

Now let's load the 256-dimensional embeddings and create a search service to explore them.


In [ ]:
!pip install scikit-learn
!pip install sentence-transformers

  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
   

In [14]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from pathlib import Path

# Load embeddings
print("Loading embeddings...")
embeddings_path = Path.cwd().parent.parent / "data" / "embeddings_256d.npy"
embeddings = np.load(embeddings_path)
print(f"✓ Loaded embeddings: {embeddings.shape}")
print(f"  - Number of plays: {embeddings.shape[0]:,}")
print(f"  - Embedding dimensions: {embeddings.shape[1]}")

# Normalize embeddings for cosine similarity (if not already normalized)
if not np.allclose(np.linalg.norm(embeddings, axis=1), 1.0):
    print("Normalizing embeddings...")
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / norms
    print("✓ Embeddings normalized")


/Users/pooks/Dev/crate/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embeddings...
✓ Loaded embeddings: (2193235, 256)
  - Number of plays: 2,193,235
  - Embedding dimensions: 256
Normalizing embeddings...
✓ Embeddings normalized


In [15]:
# Load play metadata from database
print("Loading play metadata...")
plays_df = db.query("""
    SELECT 
        id, artist, song, album, airdate, 
        labels, rotation_status, is_local, 
        is_live, is_request, comment, show
    FROM fact_plays
    WHERE artist IS NOT NULL AND song IS NOT NULL
    ORDER BY id
""")

print(f"✓ Loaded {len(plays_df):,} plays")

# Verify alignment
if len(plays_df) != len(embeddings):
    print(f"⚠️  WARNING: Mismatch! Plays: {len(plays_df)}, Embeddings: {len(embeddings)}")
    # Align them by taking the minimum
    min_len = min(len(plays_df), len(embeddings))
    plays_df = plays_df.iloc[:min_len].reset_index(drop=True)
    embeddings = embeddings[:min_len]
    print(f"✓ Aligned to {min_len:,} records")
else:
    print("✓ Embeddings and plays are aligned")

# Display sample
print("\nSample plays:")
display(plays_df.head())


Loading play metadata...
✓ Loaded 2,193,235 plays
✓ Embeddings and plays are aligned

Sample plays:


,id,artist,song,album,airdate,labels,rotation_status,is_local,is_live,is_request,comment,show
0,1,Peter Gabriel,Games Without Frontiers,Shaking the Tree: 16 Golden Greats,2017-02-08T07:34:23-08:00,"[""Virgin Japan""]",Library,0,0,0,by request!,1
1,2,New Order,Ceremony,Substance 1987,2017-02-08T07:39:31-08:00,"[""Qwest Records""]",Library,0,0,0,Requested by Kevin in memory of his friend Toby,1
2,3,Adorable,Homeboy,Against Perfection,2017-02-08T07:43:40-08:00,"[""SBK Records""]",Library,0,0,0,"English band, Adorable, formed in September 1990. Prior to that the band was known as The Candy Thieves. Adorable is a much sweeter name than The Candy Thieves.",1
3,5,Altered Images,Happy Birthday,Happy Birthday,2017-02-08T07:49:12-08:00,"[""Epic""]",Library,0,0,1,A big Happy Birthday going out to Regan from Fran!,1
4,6,Diet Cig,Tummy Ache,Swear I'm Good At This,2017-02-08T07:54:02-08:00,"[""Frenchkiss Records""]",Medium,0,0,0,None,1


In [17]:
# Load the embedding model and PCA transformer
# The embeddings were PCA-reduced from 768 to 256 dimensions
# We need to load the original model and the PCA transformer to encode queries

import json

print("Loading embedding model and PCA transformer...")

# Load metadata to get original model info
metadata_path = Path.cwd().parent.parent / "data" / "metadata.json"
if metadata_path.exists():
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    original_model_name = metadata.get('model_name', 'sentence-transformers/multi-qa-mpnet-base-dot-v1')
    original_dim = metadata.get('embedding_dimension', 768)
    print(f"✓ Loaded metadata:")
    print(f"   Original model: {original_model_name}")
    print(f"   Original dimension: {original_dim}")
    print(f"   Reduced dimension: {embeddings.shape[1]}")
else:
    # Fallback to known values from metadata.json
    original_model_name = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'
    original_dim = 768
    print(f"⚠️  metadata.json not found, using defaults:")
    print(f"   Original model: {original_model_name}")
    print(f"   Original dimension: {original_dim}")

# Load the original model
try:
    model = SentenceTransformer(original_model_name)
    model_dim = model.get_sentence_embedding_dimension()
    print(f"✓ Original model loaded: {model_dim} dimensions")
    
    if model_dim != original_dim:
        print(f"⚠️  Warning: Model dimension ({model_dim}) doesn't match metadata ({original_dim})")
except Exception as e:
    print(f"✗ Could not load model: {e}")
    model = None

# Try to load PCA transformer
pca = None
pca_path = Path.cwd().parent.parent / "data" / "pca_256.pkl"
if pca_path.exists():
    try:
        import pickle
        with open(pca_path, 'rb') as f:
            pca = pickle.load(f)
        print(f"✓ PCA transformer loaded: {pca.n_components_} components")
    except Exception as e:
        print(f"⚠️  Could not load PCA transformer: {e}")
        print("   You'll need to fit a new PCA or use the original 768-dim embeddings")
else:
    print(f"⚠️  PCA transformer not found at {pca_path}")
    print("   To enable query encoding, you need to:")
    print("   1. Fit PCA on a sample of original embeddings")
    print("   2. Save it as pca_256.pkl")
    print("   3. Or use the original 768-dim embeddings directly")


Loading embedding model and PCA transformer...
✓ Loaded metadata:
   Original model: sentence-transformers/multi-qa-mpnet-base-dot-v1
   Original dimension: 768
   Reduced dimension: 256
✓ Original model loaded: 768 dimensions
⚠️  PCA transformer not found at /Users/pooks/Dev/crate/data/pca_256.pkl
   To enable query encoding, you need to:
   1. Fit PCA on a sample of original embeddings
   2. Save it as pca_256.pkl
   3. Or use the original 768-dim embeddings directly


In [ ]:
class SemanticSearchService:
    """Semantic search service for music plays using embeddings."""
    
    def __init__(self, embeddings, plays_df, model=None, pca=None):
        """
        Args:
            embeddings: numpy array of shape (n_plays, embedding_dim)
            plays_df: DataFrame with play metadata (must have 'id' column)
            model: Optional SentenceTransformer model for encoding queries
            pca: Optional PCA transformer to reduce query embeddings to match stored embeddings
        """
        self.embeddings = embeddings
        self.plays_df = plays_df.reset_index(drop=True)
        self.model = model
        self.pca = pca
        
        # Verify alignment
        assert len(self.embeddings) == len(self.plays_df), \
            f"Mismatch: {len(self.embeddings)} embeddings vs {len(self.plays_df)} plays"
        
        print(f"✓ Search service initialized with {len(self.embeddings):,} plays")
        if self.pca is not None:
            print(f"   PCA transformer available: {self.pca.n_components_} components")
    
    def encode_query(self, query: str) -> np.ndarray:
        """Encode a text query to embedding vector, applying PCA if needed."""
        if self.model is None:
            raise ValueError("No model available. Provide a SentenceTransformer model.")
        
        # Encode and normalize with original model
        query_emb = self.model.encode([query], normalize_embeddings=True)[0]
        
        # Apply PCA reduction if available and needed
        if self.pca is not None:
            # PCA expects 2D array
            query_emb = self.pca.transform(query_emb.reshape(1, -1))[0]
            # Normalize after PCA
            query_emb = query_emb / np.linalg.norm(query_emb)
        
        # Verify final dimension matches stored embeddings
        if query_emb.shape[0] != self.embeddings.shape[1]:
            raise ValueError(
                f"Final embedding dimension ({query_emb.shape[0]}) doesn't match "
                f"stored embedding dimension ({self.embeddings.shape[1]})"
            )
        
        return query_emb
    
    def search(self, query: str, top_k: int = 10, 
               filters: dict = None, use_model: bool = True) -> pd.DataFrame:
        """
        Search for similar plays.
        
        Args:
            query: Search query text
            top_k: Number of results to return
            filters: Optional dict of filters (e.g., {'is_local': 1})
            use_model: If True, encode query with model. If False, use query as-is (for testing)
            
        Returns:
            DataFrame with results including similarity scores
        """
        # Encode query
        if use_model and self.model is not None:
            try:
                query_emb = self.encode_query(query)
            except Exception as e:
                print(f"⚠️  Could not encode with model: {e}")
                print("   Returning empty results")
                return pd.DataFrame()
        else:
            # For testing: use a random embedding or first embedding
            # In production, you'd need the model
            print("⚠️  No model available, using placeholder")
            query_emb = self.embeddings[0]  # Placeholder
        
        # Compute cosine similarities (embeddings should be normalized)
        similarities = cosine_similarity([query_emb], self.embeddings)[0]
        
        # Get top indices
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        # Build results
        results = []
        for idx in top_indices:
            play = self.plays_df.iloc[idx]
            result = {
                'rank': len(results) + 1,
                'similarity': similarities[idx],
                'id': play['id'],
                'artist': play['artist'],
                'song': play['song'],
                'album': play.get('album', ''),
                'airdate': play.get('airdate', ''),
                'labels': play.get('labels', ''),
                'rotation_status': play.get('rotation_status', ''),
                'is_local': play.get('is_local', 0),
                'comment': play.get('comment', '')[:100] if pd.notna(play.get('comment')) else ''
            }
            results.append(result)
        
        results_df = pd.DataFrame(results)
        
        # Apply filters if provided
        if filters:
            for key, value in filters.items():
                if key in results_df.columns:
                    results_df = results_df[results_df[key] == value]
        
        return results_df
    
    def search_by_play_id(self, play_id: int, top_k: int = 10) -> pd.DataFrame:
        """Find similar plays to a given play ID."""
        # Find the play index
        play_idx = self.plays_df[self.plays_df['id'] == play_id].index
        if len(play_idx) == 0:
            return pd.DataFrame()
        
        play_idx = play_idx[0]
        play_emb = self.embeddings[play_idx]
        
        # Compute similarities
        similarities = cosine_similarity([play_emb], self.embeddings)[0]
        
        # Get top indices (excluding the query play itself)
        top_indices = np.argsort(similarities)[-top_k-1:][::-1]
        top_indices = [idx for idx in top_indices if idx != play_idx][:top_k]
        
        # Build results
        results = []
        for idx in top_indices:
            play = self.plays_df.iloc[idx]
            results.append({
                'rank': len(results) + 1,
                'similarity': similarities[idx],
                'id': play['id'],
                'artist': play['artist'],
                'song': play['song'],
                'album': play.get('album', ''),
                'airdate': play.get('airdate', ''),
                'labels': play.get('labels', ''),
                'rotation_status': play.get('rotation_status', ''),
            })
        
        return pd.DataFrame(results)

# Initialize search service with model and PCA transformer
search_service = SemanticSearchService(embeddings, plays_df, model=model, pca=pca)


## 12. Test Semantic Search

Let's test the search service with some example queries. Note: If the model dimensions don't match, you'll need to use the same model that generated the embeddings.


In [ ]:
# Test search function
def test_search(query: str, top_k: int = 10, filters: dict = None):
    """Test search and display results nicely."""
    print(f"\n{'='*80}")
    print(f"Query: '{query}'")
    print(f"{'='*80}\n")
    
    try:
        results = search_service.search(query, top_k=top_k, filters=filters)
        
        if len(results) == 0:
            print("No results found")
            return
        
        for _, row in results.iterrows():
            print(f"{row['rank']:2d}. [{row['similarity']:.3f}] {row['artist']} - {row['song']}")
            if pd.notna(row['album']) and row['album']:
                print(f"     Album: {row['album']}")
            if pd.notna(row['labels']) and row['labels']:
                print(f"     Label: {row['labels']}")
            if pd.notna(row['rotation_status']):
                print(f"     Rotation: {row['rotation_status']}")
            if row['is_local'] == 1:
                print(f"     🏠 Local artist")
            if pd.notna(row['comment']) and row['comment']:
                print(f"     Comment: {row['comment']}")
            print()
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

# Example searches (will work if model dimensions match)
# If model doesn't match, you'll need to use search_by_play_id instead
print("Testing search service...")
print("\nNote: If model dimensions don't match embeddings, use search_by_play_id() instead")


In [ ]:
# Alternative: Search by similar play (works without model)
# Find a play first
sample_play = plays_df.iloc[1000]  # Pick a random play
print(f"Finding plays similar to: {sample_play['artist']} - {sample_play['song']}")
print(f"Play ID: {sample_play['id']}\n")

similar = search_service.search_by_play_id(sample_play['id'], top_k=10)
display(similar)


In [ ]:
# Try text search if model is available and dimensions match
if model is not None:
    try:
        test_search("psychedelic rock with experimental sounds", top_k=5)
    except Exception as e:
        print(f"Text search failed: {e}")
        print("\nThis is expected if the model dimensions don't match the embeddings.")
        print("You'll need to use the same model that generated the embeddings.")
else:
    print("No model available for text search. Use search_by_play_id() instead.")


## 13. Search Service Summary

The search service is now set up! You can:

1. **Search by text query** (if model dimensions match):
   ```python
   results = search_service.search("chill electronic music", top_k=10)
   ```

2. **Find similar plays** (works without model):
   ```python
   similar = search_service.search_by_play_id(play_id, top_k=10)
   ```

3. **Apply filters**:
   ```python
   results = search_service.search("indie rock", filters={'is_local': 1})
   ```

**Setup complete:**
- ✓ Original model loaded: `sentence-transformers/multi-qa-mpnet-base-dot-v1` (768 dims)
- ✓ Embeddings are PCA-reduced from 768 to 256 dimensions
- ✓ Search service supports query encoding with PCA transformation
- ⚠️  If PCA transformer (`pca_256.pkl`) is not found, text search queries won't work
  - You can still use `search_by_play_id()` for similarity search

**Next steps:**
- Ensure `pca_256.pkl` exists in the data folder for text search queries
- Add more sophisticated filtering and ranking
- Build interactive search interface
